In [1]:
"""
geonamesのデータから日本語の地名を抽出するPythonスクリプト
"""

'\ngeonamesのデータから日本語の地名を抽出するPythonスクリプト\n'

In [2]:
# Modules
import pandas as pd
import re
import regex
import pickle
from IPython.core.display import display


In [3]:
data_path = "./data/geonames/JP/JP.tsv"
# downloaded from http://download.geonames.org/export/dump/JP.zip
columns_name =[
    "geonameid",
    "name",
    "asciiname",
    "alternatenames",
    "latitude",
    "longitude",
    "feature_class",
    "feature_code",
    "country code",
    "cc2",
    "admin1 code",
    "admin2 code",
    "admin3 code",
    "admin4 code",
    "population",
    "elevation",
    "dem",
    "timezone",
    "modification date"
]

"""
geonameid         : integer id of record in geonames database
name              : name of geographical point (utf8) varchar(200)
asciiname         : name of geographical point in plain ascii characters, varchar(200)
alternatenames    : alternatenames, comma separated, ascii names automatically transliterated, convenience attribute from alternatename table, varchar(10000)
latitude          : latitude in decimal degrees (wgs84)
longitude         : longitude in decimal degrees (wgs84)
feature class     : see http://www.geonames.org/export/codes.html, char(1)
feature code      : see http://www.geonames.org/export/codes.html, varchar(10)
country code      : ISO-3166 2-letter country code, 2 characters
cc2               : alternate country codes, comma separated, ISO-3166 2-letter country code, 200 characters
admin1 code       : fipscode (subject to change to iso code), see exceptions below, see file admin1Codes.txt for display names of this code; varchar(20)
admin2 code       : code for the second administrative division, a county in the US, see file admin2Codes.txt; varchar(80) 
admin3 code       : code for third level administrative division, varchar(20)
admin4 code       : code for fourth level administrative division, varchar(20)
population        : bigint (8 byte int) 
elevation         : in meters, integer
dem               : digital elevation model, srtm3 or gtopo30, average elevation of 3''x3'' (ca 90mx90m) or 30''x30'' (ca 900mx900m) area in meters, integer. srtm processed by cgiar/ciat.
timezone          : the iana timezone id (see file timeZone.txt) varchar(40)
modification date : date of last modification in yyyy-MM-dd format
"""

"\ngeonameid         : integer id of record in geonames database\nname              : name of geographical point (utf8) varchar(200)\nasciiname         : name of geographical point in plain ascii characters, varchar(200)\nalternatenames    : alternatenames, comma separated, ascii names automatically transliterated, convenience attribute from alternatename table, varchar(10000)\nlatitude          : latitude in decimal degrees (wgs84)\nlongitude         : longitude in decimal degrees (wgs84)\nfeature class     : see http://www.geonames.org/export/codes.html, char(1)\nfeature code      : see http://www.geonames.org/export/codes.html, varchar(10)\ncountry code      : ISO-3166 2-letter country code, 2 characters\ncc2               : alternate country codes, comma separated, ISO-3166 2-letter country code, 200 characters\nadmin1 code       : fipscode (subject to change to iso code), see exceptions below, see file admin1Codes.txt for display names of this code; varchar(20)\nadmin2 code       

In [4]:
# 元データ読み込み
df = pd.read_csv(data_path,header=None,sep='\t',names=columns_name)


C:\Users\10012943\Anaconda3\envs\spatial-test\lib\site-packages\IPython\core\interactiveshell.py:3058: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [5]:
# 元データの概要表示
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99773 entries, 0 to 99772
Data columns (total 19 columns):
geonameid            99773 non-null int64
name                 99773 non-null object
asciiname            99773 non-null object
alternatenames       91481 non-null object
latitude             99773 non-null float64
longitude            99773 non-null float64
feature_class        99773 non-null object
feature_code         99772 non-null object
country code         99773 non-null object
cc2                  5015 non-null object
admin1 code          99697 non-null float64
admin2 code          80694 non-null float64
admin3 code          31467 non-null float64
admin4 code          139 non-null float64
population           99773 non-null int64
elevation            601 non-null float64
dem                  99773 non-null int64
timezone             99767 non-null object
modification date    99773 non-null object
dtypes: float64(7), int64(3), object(9)
memory usage: 14.5+ MB


In [6]:
df.head()


,geonameid,name,asciiname,alternatenames,latitude,longitude,feature_class,feature_code,country code,cc2,admin1 code,admin2 code,admin3 code,admin4 code,population,elevation,dem,timezone,modification date
0,1847943,Yotsuhama,Yotsuhama,Yotsuhama,33.41604,132.19325,P,PPL,JP,JP,5.0,1855138.0,38442.0,NaN,0,NaN,40,Asia/Tokyo,2017-04-09
1,1847944,Tahara-mura,Tahara-mura,"Tahara-mura,Tawara-mura",35.45000,136.93333,A,ADMD,JP,JP,0.0,NaN,NaN,NaN,0,NaN,106,Asia/Tokyo,2010-08-03
2,1847945,Tatsuno-shi,Tatsuno-shi,"Tatsuno,Tatsuno-cho,Tatsuno-chō,Tatsuno-machi,...",34.88804,134.51910,A,ADM2,JP,NaN,13.0,1847945.0,NaN,NaN,79870,NaN,250,Asia/Tokyo,2017-07-21
3,1847946,Shimo-funaka-mura,Shimo-funaka-mura,Shimo-funaka-mura,35.28333,139.16667,A,ADMD,JP,JP,0.0,NaN,NaN,NaN,0,NaN,19,Asia/Tokyo,2010-08-03
4,1847947,Shingū,Shingu,"Schingu,Shingu,Shingui,Shingū,Sing,Singu,Singu...",33.73333,135.98333,P,PPLA2,JP,NaN,43.0,1852105.0,NaN,NaN,31619,NaN,7,Asia/Tokyo,2017-07-22


In [7]:
# カンマ区切りの文字列から単語を抽出し、日本語の単語のみ抽出
def extract_JPwords_old(words):
    words = str(words) # 型を文字列にする(nanだとfloatになるため)
    list_words = words.split(',')
    list_kanjiwords = []
    for word in list_words:
        # 漢字、ひらがな、カタカナのいずれかを含むwordを抽出
        #kanjiword = regex.findall(r'\p{Script=Han}+',word)
        kanjiword = regex.findall(r'[\p{Script=Han}|\p{Script=Hiragana}|\p{Script=Katakana}]+',word)
        if len(kanjiword)>0:
            list_kanjiwords.append(kanjiword[0])
    
    return ','.join(list_kanjiwords)



In [8]:
def extract_JPwords(words):
    """カンマ区切りの文字列から日本語の単語を抽出する
    
    Arguments:
        words {[string]} -- [カンマ区切りで多様な言語の地名を連ねた文字列。
        例:'Acugi,Atsugi,Atsugicho,Atsugichō,Atsuki,Atugi,Atugi-chhi,Atugi-chhī,NJA,asseugi si,atswghy,atswgy  kanagawa,hou mu,hou mu ding,hou mu shi,xa sungi,Атсуги,Ацуги,Ацуґі,آتسوگی، کاناگاوا,أتسوغي,اتسوگی، کاناگاوا,อะสึงิ,厚木,厚木市,厚木町,아쓰기 시']
    
    Returns:
        [string] -- [カンマ区切りの文字列で日本語の単語のみ抽出したもの。
        例:"厚木,厚木市,厚木町"]
    """

    words = str(words) # 型を文字列にする(nanだとfloatになるため)
    list_words = words.split(',')
    list_kanjiwords = []
    list_hiraganawords = []
    list_katakanawords = []
    for word in list_words:
        # 全角空白が含まれている場合は削除する
        word = word.replace("\u3000","")

        # 半角スペース、かつ、英数字がふくまれる場合は中国語(日本語地名でない)と判定
        result_space = re.search(r' ', word)
        result_alphabet = re.search(r'\w',word)
        if result_space and result_alphabet:
            continue
        else:
            # wordに漢字,ひらがな,カタカナを含むかを調べる
            # https://note.nkmk.me/python-re-regex-character-type/
            kanjiword = regex.findall(r'\p{Script=Han}+',word)
            hiraganaword = regex.findall(r'\p{Script=Hiragana}+',word)
            katakanaword = regex.findall(r'\p{Script=Katakana}+',word)
            
            if(len(word)>1):
                if len(kanjiword)>0:
                    list_kanjiwords.append(word)
                elif len(hiraganaword)>0:
                    list_katakanawords.append(word)
                elif len(katakanaword)>0:
                    list_katakanawords.append(word)
                else:
                    pass
    
    # 適切な日本語のwordのリストを出力
    if len(list_kanjiwords)>0: #漢字を含むwordがある場合
        return ','.join(list_kanjiwords)
    #elif len(list_hiraganawords)>0: #漢字を含むwordがなく、ひらがなのwordがある場合
    #    return ','.join(list_hiraganawords)
    #elif len(list_katakanawords)>0:  #漢字を含むword、ひらがなを含むwordがなく、カタカナのwordがある場合
    #    return ','.join(list_katakanawords)
    else:
        return ''



In [9]:
# 関数extract_JPwordsのテスト
print(extract_JPwords("totsuka,とつか,トツカ,戸塚,戸塚区,とつか区"))
print(extract_JPwords('Acugi,Atsugi,Atsugicho,Atsugichō,Atsuki,Atugi,Atugi-chhi,Atugi-chhī,NJA,asseugi si,atswghy,atswgy  kanagawa,hou mu,hou mu ding,hou mu shi,xa sungi,Атсуги,Ацуги,Ацуґі,آتسوگی، کاناگاوا,أتسوغي,اتسوگی، کاناگاوا,อะสึงิ,厚木,厚木市,厚木町,아쓰기 시'))


戸塚,戸塚区,とつか区
厚木,厚木市,厚木町


In [10]:
# 日本語の地名を表す文字列を追加
df['JPwords'] = df.alternatenames.map(extract_JPwords)
df.head(10)


,geonameid,name,asciiname,alternatenames,latitude,longitude,feature_class,feature_code,country code,cc2,admin1 code,admin2 code,admin3 code,admin4 code,population,elevation,dem,timezone,modification date,JPwords
0,1847943,Yotsuhama,Yotsuhama,Yotsuhama,33.41604,132.19325,P,PPL,JP,JP,5.0,1855138.0,38442.0,NaN,0,NaN,40,Asia/Tokyo,2017-04-09,
1,1847944,Tahara-mura,Tahara-mura,"Tahara-mura,Tawara-mura",35.45000,136.93333,A,ADMD,JP,JP,0.0,NaN,NaN,NaN,0,NaN,106,Asia/Tokyo,2010-08-03,
2,1847945,Tatsuno-shi,Tatsuno-shi,"Tatsuno,Tatsuno-cho,Tatsuno-chō,Tatsuno-machi,...",34.88804,134.51910,A,ADM2,JP,NaN,13.0,1847945.0,NaN,NaN,79870,NaN,250,Asia/Tokyo,2017-07-21,たつの市
3,1847946,Shimo-funaka-mura,Shimo-funaka-mura,Shimo-funaka-mura,35.28333,139.16667,A,ADMD,JP,JP,0.0,NaN,NaN,NaN,0,NaN,19,Asia/Tokyo,2010-08-03,
4,1847947,Shingū,Shingu,"Schingu,Shingu,Shingui,Shingū,Sing,Singu,Singu...",33.73333,135.98333,P,PPLA2,JP,NaN,43.0,1852105.0,NaN,NaN,31619,NaN,7,Asia/Tokyo,2017-07-22,"新宮,新宮市"
5,1847948,Niōji Dake,Nioji Dake,"Ninoji-dake,Ninoji-yama,Ninōji-dake,Nioji Dake...",37.90103,139.49979,T,MT,JP,NaN,29.0,1852604.0,NaN,NaN,0,NaN,1381,Asia/Tokyo,2017-04-09,二王子岳
6,1847949,Naha Air Base,Naha Air Base,"AHA,Naha Air Base,Naha Air Force Base,Naha Kuk...",26.19796,127.64570,S,AIRB,JP,NaN,47.0,1856030.0,NaN,NaN,0,NaN,4,Asia/Tokyo,2017-04-09,
7,1847950,Naegi-chō,Naegi-cho,"Naegi-cho,Naegi-chō,Naegi-machi,Naeki-machi",35.51667,137.48333,A,ADMD,JP,NaN,0.0,NaN,NaN,NaN,0,NaN,402,Asia/Tokyo,2010-08-03,
8,1847951,Mimitsu,Mimitsu,"Mimitsu,Mimizu",32.33333,131.60000,P,PPL,JP,JP,25.0,1862036.0,NaN,NaN,0,NaN,46,Asia/Tokyo,2017-04-09,
9,1847952,Kujū-san,Kuju-san,"Kuju-san,Kuju-sani,Kuju-zan,Kujū-san,Kujū-sani...",33.11667,131.23333,T,MTS,JP,JP,0.0,NaN,NaN,NaN,0,NaN,1041,Asia/Tokyo,2012-01-19,


In [11]:
# 日本語の地名をリスト化する
list_placename =[]
for words in df[df['JPwords']!=""].JPwords:
    for word in words.split(','):
        list_placename.append(word)

# 重複する語を削除するためセットにする
set_placename = set(list_placename)


In [12]:
# セットをファイルに保存
with open('./data/placenameSet.pkl','wb') as f:
    pickle.dump(set_placename,f)


In [13]:
# ファイル読み込みのテスト
with open('./data/placenameSet.pkl','rb') as f_test:
    set_placename_test = pickle.load(f_test)


In [14]:
display(set_placename_test)


{'掃部ヶ岳',
 '更村',
 '本谷',
 '笹ガ陣',
 '千里内',
 '高柳町門出',
 '沖ノ山',
 '玉之浦湾',
 '那智駅',
 '真方',
 '田川郡',
 '中村区役所',
 '森ケ崎観音堂',
 '下染田',
 '桑折町役場',
 '相良',
 '共励橋',
 '中野平',
 '宿毛',
 '岩神',
 '筑穂元吉',
 '鯎川',
 '上湧別屯田市街地',
 '大倉森',
 '亀岡',
 '間細工',
 '田貫湖',
 '鵜の鳥岩',
 '萩ヶ岡',
 '西牟婁郡',
 '寺畑',
 'つつじ台',
 '上八色',
 '永瀬ダム',
 '三国町横越',
 '江戸川台駅',
 '杉ヶ崎',
 '中の橋',
 'オサッペ沢川',
 '福門',
 '長井一丁目',
 '杉岳',
 '藤ノ木駅',
 '羽保屋山',
 'しらかわ台団地',
 '綾北川',
 '角田浜',
 '下北手梨平',
 '大喰岳',
 '浜北区',
 '阿蘇ミルク牧場',
 '智西',
 '船小屋駅',
 '黒峰',
 '箕子橋',
 '北廣島市',
 '日野郡',
 '向外瀬',
 '申ヶ野',
 '南陵町',
 '对马岛',
 '小佐々町岳ノ木場',
 '霧島市役所',
 '竹辺町',
 '篠原池',
 '柏原',
 '切畑ダム',
 '江迎湾',
 '人吉城跡',
 '小路口',
 '美芳',
 '岐阜県',
 '郷戸駅',
 '真浦',
 '羽茂滝平',
 '信楽町黄瀬',
 '東田川郡',
 '野尻駅',
 '唐橋前駅',
 '御嶽駅',
 '八幡山法蓮寺',
 '馬田',
 '太田山',
 '東曲渕',
 '青谷駅',
 '大原山',
 '亀山市',
 '上東莱川',
 '弐町',
 '番田',
 '鳥原',
 '公立病院前駅',
 '江奈湾',
 '山の手台',
 '更埴',
 '京浜運河',
 '繋沢',
 '本荘駅',
 '西勇足',
 '渕沢',
 '太平中関',
 '大元駅',
 '箱根山',
 '三丁',
 '三ケ',
 '三河鳥羽駅',
 '支幌呂西',
 'カレホコ鼻',
 '笹木野駅',
 '石狩月形駅',
 '農進',
 '柳沢駅',
 '柿木町',
 '黒園山',
 '大野台',
 '小糸大谷',
 '高松山',
 '鴫ケ沢川',
 '伊那